# Transportation Problem — Formulation and Solution (Julia)

> **ORKit Free** · Linear Programming (LP) · JuMP + HiGHS

---

## The Problem

Ship goods from $m$ suppliers to $n$ customers, each with limited supply and required demand,
**minimizing total transportation cost**.

### Formulation

$$\min \quad \sum_{i} \sum_{j} c_{ij} \, x_{ij}$$

$$\text{s.t.} \quad \sum_{j} x_{ij} \leq s_i \quad \forall i \quad \text{(supply)}$$

$$\sum_{i} x_{ij} \geq d_j \quad \forall j \quad \text{(demand)}$$

$$x_{ij} \geq 0$$

In [ ]:
using JuMP
using HiGHS
using JSON3

println("JuMP version: ", pkgversion(JuMP))

## Loading the Instance

In [ ]:
data = JSON3.read(read("../instances/small_3x4.json", String))

suppliers = data.suppliers
customers = data.customers
m = length(suppliers)
n = length(customers)
s = [sup.supply for sup in suppliers]
d = [cus.demand for cus in customers]
c = [Float64(data.costs[i][j]) for i in 1:m, j in 1:n]

println("Suppliers: $m  |  Customers: $n")
println("Total supply: $(sum(s))  |  Total demand: $(sum(d))")
println()
println("Cost matrix:")
for i in 1:m
    println("  S$i: ", join([@sprintf("%5.0f", c[i,j]) for j in 1:n], " "))
end

## Building and Solving the JuMP Model

In [ ]:
using Printf

model = Model(HiGHS.Optimizer)
set_silent(model)

# ── Variables ─────────────────────────────────────
@variable(model, x[1:m, 1:n] >= 0)

# ── Objective ─────────────────────────────────────
@objective(model, Min, sum(c[i, j] * x[i, j] for i in 1:m, j in 1:n))

# ── Supply constraints ────────────────────────────
@constraint(model, supply[i in 1:m], sum(x[i, j] for j in 1:n) <= s[i])

# ── Demand constraints ────────────────────────────
@constraint(model, demand[j in 1:n], sum(x[i, j] for i in 1:m) >= d[j])

optimize!(model)

println("Status    : ", termination_status(model))
println("Total cost: ", round(objective_value(model); digits=2))

## Shipping Plan

In [ ]:
println("Shipping Plan:")
println()
header = "       " * join([@sprintf("C%d%7s", j, "") for j in 1:n]) * "  | Shipped  Supply"
println(header)
println("-" ^ length(header))

for i in 1:m
    shipped = 0.0
    row = @sprintf("S%d     ", i)
    for j in 1:n
        val = value(x[i, j])
        shipped += val
        row *= @sprintf("%8.1f", val)
    end
    row *= @sprintf("  | %7.1f  %6.0f", shipped, s[i])
    println(row)
end

println()
received = [sum(value(x[i, j]) for i in 1:m) for j in 1:n]
println("Received: ", join([@sprintf("%8.1f", received[j]) for j in 1:n]))
println("Demand:   ", join([@sprintf("%8.0f", d[j]) for j in 1:n]))

## Shadow Prices (Dual Values)

In [ ]:
println("Supply shadow prices:")
for i in 1:m
    println("  Supplier $i: ", round(dual(supply[i]); digits=2))
end

println()
println("Demand shadow prices:")
for j in 1:n
    println("  Customer $j: ", round(dual(demand[j]); digits=2))
end

## Key Takeaways

- The transportation problem has a **totally unimodular** constraint matrix,
  so LP relaxation always gives integer solutions
- Shadow prices reveal the economic value of expanding supply or reducing demand
- JuMP makes it easy to access dual values via `dual(constraint_ref)`

### References

- Hitchcock, F. L. (1941). The distribution of a product from several sources to numerous localities.
- Hillier, F. S., Lieberman, G. J. (2014). *Introduction to OR* (10th ed.). Ch. 8.